# Classical shadows — many observables from one set of measurements

Key idea: a single batch of randomized single-qubit measurements ("snapshots") of a state
lets us estimate **many** observables at once. We take shadow snapshots of a generic
entangled 5-qubit state, estimate **all 1- and 2-local Pauli observables** (105 of them) from
the *same* snapshots, and compare to the exact values.

If shadows work, every point lands on the diagonal $y=x$ within its error bar.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt

from pai_shadow.backend import Circuit, get_backend
from pai_shadow.classical_shadow import ClassicalShadow

In [ ]:
n_qubits = 5
backend = get_backend("qulacs")

# a generic entangled state: varied single-qubit rotations + nearest-neighbour entanglers
state = Circuit(n_qubits)
for q in range(n_qubits):
    state.ry(q, 0.5 + 0.55 * q)
for q in range(n_qubits - 1):
    state.rzz(q, q + 1, 0.7)
for q in range(n_qubits):
    state.rx(q, 0.25)

# all 1- and 2-local Pauli observables
def label(positions, types):
    s = ["I"] * n_qubits
    for p, t in zip(positions, types):
        s[p] = t
    return "".join(s)

observables = []
for q in range(n_qubits):
    observables += [label([q], [P]) for P in "XYZ"]
for a, b in itertools.combinations(range(n_qubits), 2):
    observables += [label([a, b], [P, Q]) for P in "XYZ" for Q in "XYZ"]
locality = np.array([n_qubits - o.count("I") for o in observables])  # 1 or 2
print(len(observables), "observables (1- and 2-local)")

In [ ]:
# one batch of snapshots -> estimate every observable from it
N = 6000
shadow = ClassicalShadow(backend=backend, seed=0)
factors = shadow.snapshots(state, N)

PIDX = {"X": 0, "Y": 1, "Z": 2}
def per_snapshot(P):
    v = np.ones(N)
    for q, p in enumerate(P):
        if p != "I":
            v *= factors[:, q, PIDX[p]]
    return v

shadow_mean = np.array([per_snapshot(P).mean() for P in observables])
shadow_err  = np.array([per_snapshot(P).std() / np.sqrt(N) for P in observables])
exact = np.array([backend.expectation(state, P) for P in observables])

rms = np.sqrt(np.mean((shadow_mean - exact) ** 2))
print(f"RMS(shadow - exact) over {len(observables)} observables = {rms:.3f}")

In [ ]:
lim = 1.05
plt.figure(figsize=(6.5, 6.5))
plt.plot([-lim, lim], [-lim, lim], color="gray", lw=1, label="$y=x$")
for k, color, name in [(1, "tab:blue", "1-local"), (2, "tab:red", "2-local")]:
    m = locality == k
    plt.scatter(exact[m], shadow_mean[m], s=55, c=color, edgecolors="black",
                linewidths=0.4, alpha=0.85, label=name)
plt.xlabel(r"exact $\langle P\rangle$"); plt.ylabel("shadow estimate")
plt.title(f"{len(observables)} Pauli observables from {N} snapshots")
plt.legend(); plt.tight_layout(); plt.show()

## Convergence: error shrinks as $1/\sqrt{N}$

Watch a few observables converge to their exact values (dotted lines) as the number of
snapshots grows.

In [ ]:
probe = ["ZIIII", "ZZIII", "XXIII"]
Ns = np.unique(np.logspace(1.5, np.log10(N), 12).astype(int))
plt.figure(figsize=(8, 4.5))
for P, color in zip(probe, ["tab:blue", "tab:red", "tab:green"]):
    v = per_snapshot(P)
    means = [v[:m].mean() for m in Ns]
    errs  = [v[:m].std() / np.sqrt(m) for m in Ns]
    plt.errorbar(Ns, means, yerr=2 * np.array(errs), fmt="o-", ms=3, capsize=2,
                 color=color, label=rf"$\langle {P} \rangle$")
    plt.axhline(backend.expectation(state, P), color=color, ls=":", lw=1)
plt.xscale("log"); plt.xlabel("number of snapshots $N$"); plt.ylabel("estimate")
plt.title("Convergence to the exact value (dotted lines)")
plt.legend(); plt.tight_layout(); plt.show()

## Takeaways

- **One** batch of randomized measurements estimates **all** 105 observables — that is the point of classical shadows.
- Every estimate lands on $y=x$ within $\pm2\sigma$, and the error shrinks as $1/\sqrt{N}$.
- $k$-local observables carry a variance factor $3^k$, so 2-local points scatter a little more than 1-local.